# Module 38: Compiled Autograd & AOTAutograd
## Optimizing the Backward Pass

**Prerequisites**: Modules 03 (Autograd), 04 (Neural Networks), 08 (torch.compile), 35 (The Dispatcher)

In this notebook we explore how PyTorch compiles the backward pass alongside the forward.

Key topics:
- Why eager backward is slow
- AOTAutograd: tracing forward + backward at compile time
- The joint graph and min-cut partitioning
- Compiled Autograd: compiling the autograd engine
- Debugging and inspecting compiled graphs

In [ ]:
import torch
import torch.nn as nn
import time

print(f"PyTorch version: {torch.__version__}")
print(f"Device: CPU (all examples run on CPU)")

## 1. Why Backward is Slow in Eager Mode

In eager mode, the backward pass executes ops one-by-one through the C++ autograd engine.
Each op has kernel launch overhead. `torch.compile` optimizes the forward, but by default
the backward is still eager.

```
Forward:   [Compiled — fused, optimized, fast]
Backward:  [Eager — one op at a time, no fusion]
```

## 2. Standard Autograd: The grad_fn Chain

Let's see how eager autograd builds the backward graph at runtime.

In [ ]:
x = torch.randn(3, 4, requires_grad=True)
weight = torch.randn(4, 2, requires_grad=True)
bias = torch.randn(2, requires_grad=True)

# Forward pass builds grad_fn chain at runtime
mm = x @ weight
added = mm + bias
activated = torch.relu(added)
loss = activated.sum()

# Walk the grad_fn chain
print("grad_fn chain (built at runtime):")
fn = loss.grad_fn
depth = 0
while fn is not None:
    indent = "  " * depth
    print(f"{indent}-> {fn.__class__.__name__}")
    if hasattr(fn, 'next_functions') and fn.next_functions:
        fn = fn.next_functions[0][0]
    else:
        fn = None
    depth += 1

loss.backward()
print(f"\nx.grad shape: {x.grad.shape}")
print(f"weight.grad shape: {weight.grad.shape}")

## 3. AOTAutograd: Trace Forward + Backward at Compile Time

Instead of building the backward graph at runtime, AOTAutograd traces **both** forward and
backward at compile time. This produces two FX graphs that both get optimized by Inductor.

```
User Model → [AOTAutograd] → Forward Graph (FX) + Backward Graph (FX)
                                      ↓                    ↓
                                [Inductor]           [Inductor]
                                      ↓                    ↓
                               Optimized Fwd        Optimized Bwd
```

## 4. aot_function: Inspect Both Graphs

Using `aot_function` with custom compilers to see the forward and backward graphs.

In [ ]:
from torch._functorch.aot_autograd import aot_function

def fw_compiler(gm, example_inputs):
    print("=" * 50)
    print("FORWARD GRAPH:")
    print("=" * 50)
    gm.graph.print_tabular()
    n_ops = len([n for n in gm.graph.nodes if n.op == 'call_function'])
    print(f"Forward op count: {n_ops}")
    return gm

def bw_compiler(gm, example_inputs):
    print("\n" + "=" * 50)
    print("BACKWARD GRAPH:")
    print("=" * 50)
    gm.graph.print_tabular()
    n_ops = len([n for n in gm.graph.nodes if n.op == 'call_function'])
    print(f"Backward op count: {n_ops}")
    return gm

def simple_fn(x, weight, bias):
    return torch.relu(x @ weight + bias)

compiled = aot_function(simple_fn, fw_compiler=fw_compiler, bw_compiler=bw_compiler)

x = torch.randn(4, 8, requires_grad=True)
w = torch.randn(8, 3, requires_grad=True)
b = torch.randn(3, requires_grad=True)

out = compiled(x, w, b)
print(f"\nOutput shape: {out.shape}")

out.sum().backward()
print(f"x.grad shape: {x.grad.shape}")
print(f"w.grad shape: {w.grad.shape}")

## 5. The Joint Graph and Partitioning

Before partitioning, AOTAutograd creates a **joint graph** containing both forward and
backward operations. The partitioner then splits this into two separate graphs.

In [ ]:
from torch._functorch.partitioners import default_partition

def partition_inspector(joint_module, joint_inputs, *, num_fwd_outputs):
    """Custom partitioner that inspects the joint graph."""
    print("JOINT GRAPH (forward + backward combined):")
    print(f"Number of forward outputs: {num_fwd_outputs}")
    
    call_nodes = [n for n in joint_module.graph.nodes if n.op == 'call_function']
    print(f"Total call_function nodes: {len(call_nodes)}")
    print("\nOperations in joint graph:")
    for node in call_nodes:
        name = node.target.__name__ if hasattr(node.target, '__name__') else str(node.target)
        print(f"  {node.name}: {name}")
    
    return default_partition(joint_module, joint_inputs, num_fwd_outputs=num_fwd_outputs)

def model_fn(x, weight):
    h = x @ weight
    h = torch.relu(h)
    return h.sum()

compiled = aot_function(
    model_fn,
    fw_compiler=lambda gm, _: gm,
    bw_compiler=lambda gm, _: gm,
    partition_fn=partition_inspector,
)

x = torch.randn(4, 8, requires_grad=True)
w = torch.randn(8, 4, requires_grad=True)
out = compiled(x, w)
out.backward()
print(f"\nGradients computed. x.grad norm: {x.grad.norm():.4f}")

## 6. min_cut_rematerialization Explained

The min-cut partitioner decides which forward activations to **save** for backward vs
**recompute** during backward:

- **Save**: Expensive ops like matmul, convolution, attention
- **Recompute**: Cheap ops like relu, add, mul, sigmoid

This is automatic operator-level activation checkpointing.

In [ ]:
def count_saved_tensors(fn, *args):
    """Count tensors saved from forward for backward."""
    info = {'fw_outputs': 0}
    
    def fw_compiler(gm, example_inputs):
        output_node = [n for n in gm.graph.nodes if n.op == 'output'][0]
        output_args = output_node.args[0]
        info['fw_outputs'] = len(output_args) if isinstance(output_args, (list, tuple)) else 1
        return gm
    
    compiled = aot_function(fn, fw_compiler=fw_compiler, bw_compiler=lambda gm, _: gm)
    out = compiled(*args)
    out.backward() if isinstance(out, torch.Tensor) and out.dim() == 0 else out.sum().backward()
    
    return info['fw_outputs'] - 1  # subtract the actual model output

# Simple: linear
def linear_model(x, w, b):
    return (x @ w + b).sum()

# Medium: linear + relu
def relu_model(x, w, b):
    return torch.relu(x @ w + b).sum()

# Complex: 2-layer MLP
def mlp_model(x, w1, b1, w2, b2):
    h = torch.relu(x @ w1 + b1)
    return (h @ w2 + b2).sum()

x = torch.randn(4, 8, requires_grad=True)
w1 = torch.randn(8, 6, requires_grad=True)
b1 = torch.randn(6, requires_grad=True)
w2 = torch.randn(6, 3, requires_grad=True)
b2 = torch.randn(3, requires_grad=True)

saved_linear = count_saved_tensors(linear_model, x.clone().requires_grad_(True), w1[:, :3], b1[:3])
saved_relu = count_saved_tensors(relu_model, x.clone().requires_grad_(True), w1[:, :3], b1[:3])
saved_mlp = count_saved_tensors(mlp_model, x.clone().requires_grad_(True), w1, b1, w2, b2)

print(f"{'Architecture':<30} {'Saved Tensors':>14}")
print(f"{'─' * 30} {'─' * 14}")
print(f"{'Linear only':<30} {saved_linear:>14}")
print(f"{'Linear + ReLU':<30} {saved_relu:>14}")
print(f"{'2-layer MLP':<30} {saved_mlp:>14}")
print("\nMore layers → more saved tensors, but min-cut keeps it efficient.")

## 7. Compiled Autograd: Compile Backward Too

Compiled Autograd goes further: it compiles the autograd **engine** itself.

In [ ]:
class SmallMLP(nn.Module):
    def __init__(self, d_in, d_hidden, d_out):
        super().__init__()
        self.fc1 = nn.Linear(d_in, d_hidden)
        self.fc2 = nn.Linear(d_hidden, d_hidden)
        self.fc3 = nn.Linear(d_hidden, d_out)
    
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)

# Enable compiled autograd
torch._dynamo.config.compiled_autograd = True

model = torch.compile(SmallMLP(32, 64, 10))
x = torch.randn(16, 32)
target = torch.randn(16, 10)

# Both forward AND backward are compiled
loss = nn.functional.mse_loss(model(x), target)
loss.backward()

print(f"Loss: {loss.item():.6f}")
print(f"fc1.weight.grad norm: {model._orig_mod.fc1.weight.grad.norm():.6f}")
print("Both forward and backward compiled!")

torch._dynamo.config.compiled_autograd = False

## 8. Training Comparison: Eager vs Compiled

Verify that compiled training produces the same loss trajectory as eager.

In [ ]:
def train_loop(model, optimizer, x, y, n_steps):
    losses = []
    for _ in range(n_steps):
        optimizer.zero_grad()
        loss = nn.functional.mse_loss(model(x), y)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    return losses

torch.manual_seed(42)
d_in, d_hidden, d_out = 32, 64, 10
n_steps = 50
x_data = torch.randn(16, d_in)
y_data = torch.randn(16, d_out)

# Eager
torch.manual_seed(42)
m_eager = nn.Sequential(nn.Linear(d_in, d_hidden), nn.ReLU(), nn.Linear(d_hidden, d_out))
losses_eager = train_loop(m_eager, torch.optim.SGD(m_eager.parameters(), lr=0.01), x_data, y_data, n_steps)

# Compiled with compiled autograd
torch.manual_seed(42)
m_compiled = nn.Sequential(nn.Linear(d_in, d_hidden), nn.ReLU(), nn.Linear(d_hidden, d_out))
m_compiled = torch.compile(m_compiled)
torch._dynamo.config.compiled_autograd = True
losses_compiled = train_loop(m_compiled, torch.optim.SGD(m_compiled.parameters(), lr=0.01), x_data, y_data, n_steps)
torch._dynamo.config.compiled_autograd = False

print(f"{'Step':<8} {'Eager':>10} {'Compiled':>10} {'Diff':>12}")
print(f"{'─'*8} {'─'*10} {'─'*10} {'─'*12}")
for i in range(0, n_steps, 10):
    diff = abs(losses_eager[i] - losses_compiled[i])
    print(f"{i:<8} {losses_eager[i]:>10.4f} {losses_compiled[i]:>10.4f} {diff:>12.2e}")

max_diff = max(abs(e - c) for e, c in zip(losses_eager, losses_compiled))
print(f"\nMax loss difference: {max_diff:.2e}")
print(f"Losses match: {max_diff < 1e-4}")

## 9. Saved Tensors Analysis

Let's analyze what the min-cut partitioner saves vs recomputes across different architectures.

In [ ]:
def analyze_saved(fn, label, *args):
    """Analyze forward graph outputs (saved tensors) for a function."""
    info = {}
    
    def fw_compiler(gm, example_inputs):
        output_node = [n for n in gm.graph.nodes if n.op == 'output'][0]
        output_args = output_node.args[0]
        info['total_outputs'] = len(output_args) if isinstance(output_args, (list, tuple)) else 1
        
        call_nodes = [n for n in gm.graph.nodes if n.op == 'call_function']
        info['fw_ops'] = len(call_nodes)
        return gm
    
    def bw_compiler(gm, example_inputs):
        call_nodes = [n for n in gm.graph.nodes if n.op == 'call_function']
        info['bw_ops'] = len(call_nodes)
        return gm
    
    compiled = aot_function(fn, fw_compiler=fw_compiler, bw_compiler=bw_compiler)
    out = compiled(*args)
    if isinstance(out, torch.Tensor) and out.dim() == 0:
        out.backward()
    else:
        out.sum().backward()
    
    saved = info.get('total_outputs', 1) - 1
    print(f"  {label:<35} fwd_ops={info.get('fw_ops', '?'):>3}  bwd_ops={info.get('bw_ops', '?'):>3}  saved={saved}")
    return info

# Different architectures
def arch1(x, w):
    return (x @ w).sum()

def arch2(x, w):
    return torch.relu(x @ w).sum()

def arch3(x, w1, w2):
    return torch.relu(x @ w1 @ w2).sum()

def arch4(x, w1, w2):
    h = torch.relu(x @ w1)
    return torch.sigmoid(h @ w2).sum()

def arch5(x, w1, w2, w3):
    h = torch.relu(x @ w1)
    h = torch.sigmoid(h @ w2)
    return (h @ w3).sum()

x = torch.randn(4, 8, requires_grad=True)
w8x6 = torch.randn(8, 6, requires_grad=True)
w6x4 = torch.randn(6, 4, requires_grad=True)
w4x3 = torch.randn(4, 3, requires_grad=True)

print(f"  {'Architecture':<35} {'Fwd Ops':>8}  {'Bwd Ops':>8}  {'Saved':>5}")
print(f"  {'─'*35} {'─'*8}  {'─'*8}  {'─'*5}")
analyze_saved(arch1, "x @ w", x.clone().requires_grad_(True), w8x6)
analyze_saved(arch2, "relu(x @ w)", x.clone().requires_grad_(True), w8x6)
analyze_saved(arch3, "relu(x @ w1 @ w2)", x.clone().requires_grad_(True), w8x6, w6x4)
analyze_saved(arch4, "sigmoid(relu(x@w1) @ w2)", x.clone().requires_grad_(True), w8x6, w6x4)
analyze_saved(arch5, "relu->sigmoid->linear (3-layer)", x.clone().requires_grad_(True), w8x6, w6x4, w4x3)

## 10. Eager vs AOT Backward — Correctness

Verifying that AOT-compiled backward produces the same gradients as eager.

In [ ]:
torch.manual_seed(123)

def test_fn(x, w1, b1, w2):
    h = torch.relu(x @ w1 + b1)
    return (h @ w2).sum()

x_data = torch.randn(8, 16)
w1_data = torch.randn(16, 12)
b1_data = torch.randn(12)
w2_data = torch.randn(12, 4)

# Eager
x_e = x_data.clone().requires_grad_(True)
w1_e = w1_data.clone().requires_grad_(True)
b1_e = b1_data.clone().requires_grad_(True)
w2_e = w2_data.clone().requires_grad_(True)
test_fn(x_e, w1_e, b1_e, w2_e).backward()

# AOT
x_a = x_data.clone().requires_grad_(True)
w1_a = w1_data.clone().requires_grad_(True)
b1_a = b1_data.clone().requires_grad_(True)
w2_a = w2_data.clone().requires_grad_(True)
compiled = aot_function(test_fn, fw_compiler=lambda gm, _: gm, bw_compiler=lambda gm, _: gm)
compiled(x_a, w1_a, b1_a, w2_a).backward()

print(f"x grad match:  {torch.allclose(x_e.grad, x_a.grad)}")
print(f"w1 grad match: {torch.allclose(w1_e.grad, w1_a.grad)}")
print(f"b1 grad match: {torch.allclose(b1_e.grad, b1_a.grad)}")
print(f"w2 grad match: {torch.allclose(w2_e.grad, w2_a.grad)}")

max_diff = max(
    (x_e.grad - x_a.grad).abs().max().item(),
    (w1_e.grad - w1_a.grad).abs().max().item(),
    (b1_e.grad - b1_a.grad).abs().max().item(),
    (w2_e.grad - w2_a.grad).abs().max().item(),
)
print(f"Max gradient diff: {max_diff:.2e}")

## 11. Debugging: TORCH_LOGS

Key environment variables for debugging AOTAutograd and Compiled Autograd:

```bash
# AOTAutograd forward/backward graphs
TORCH_LOGS="aot" python script.py

# Generated Inductor code
TORCH_LOGS="output_code" python script.py

# Graph breaks
TORCH_LOGS="graph_breaks" python script.py

# Compiled autograd
TORCH_LOGS="compiled_autograd" python script.py

# Partitioner decisions
torch._functorch.config.debug_partitioner = True
```

In [ ]:
# Programmatic logging example
import logging

print("Available AOTAutograd logging:")
print("  torch._logging.set_logs(aot=logging.DEBUG)")
print("  torch._logging.set_logs(compiled_autograd=logging.DEBUG)")
print("  torch._functorch.config.debug_partitioner = True")
print()
print("These produce verbose output showing:")
print("  - Joint graph (forward + backward combined)")
print("  - Partitioning decisions (save vs recompute)")
print("  - Final forward and backward graphs")
print("  - Generated Inductor code")

## 12. AOTAutograd vs Standard Autograd

| Feature | Standard Autograd | AOTAutograd |
|---------|------------------|-------------|
| Graph built | Runtime | Compile time |
| Backward optimized | No | Yes (Inductor) |
| Memory planning | Manual (checkpointing) | Automatic (min-cut) |
| Kernel fusion | None | Yes |
| Works with compile | Forward only | Forward + backward |

## 13. Backward Op Fusion

Let's see how AOTAutograd organizes backward operations.

In [ ]:
bw_ops = {}

def bw_analyzer(gm, example_inputs):
    call_nodes = [n for n in gm.graph.nodes if n.op == 'call_function']
    for n in call_nodes:
        name = n.target.__name__ if hasattr(n.target, '__name__') else str(n.target)
        short = name.split('.')[-1]
        bw_ops[short] = bw_ops.get(short, 0) + 1
    return gm

def model_fn(x, w1, b1, w2, b2):
    h = torch.relu(x @ w1 + b1)
    h = torch.sigmoid(h @ w2 + b2)
    return h.sum()

compiled = aot_function(model_fn, fw_compiler=lambda gm, _: gm, bw_compiler=bw_analyzer)

x = torch.randn(8, 16, requires_grad=True)
w1 = torch.randn(16, 12, requires_grad=True)
b1 = torch.randn(12, requires_grad=True)
w2 = torch.randn(12, 8, requires_grad=True)
b2 = torch.randn(8, requires_grad=True)

compiled(x, w1, b1, w2, b2).backward()

print("Backward op breakdown:")
for op, count in sorted(bw_ops.items()):
    print(f"  {op}: {count}")
print(f"\nTotal backward ops: {sum(bw_ops.values())}")
print("Inductor fuses element-wise ops (sigmoid_bwd, threshold_bwd, add, mul)")
print("into fewer kernels. Matmul ops remain separate (already efficient).")

## 14. Timing Comparison

On CPU, the difference is small. On GPU, compiled autograd provides 1.2-1.5x speedup.

In [ ]:
torch.manual_seed(0)
d_in, d_hidden, d_out = 128, 256, 64
n_warmup, n_steps = 10, 100
x_data = torch.randn(64, d_in)
y_data = torch.randn(64, d_out)

def make_model():
    torch.manual_seed(0)
    return nn.Sequential(
        nn.Linear(d_in, d_hidden), nn.ReLU(),
        nn.Linear(d_hidden, d_hidden), nn.ReLU(),
        nn.Linear(d_hidden, d_out),
    )

def bench(model, label):
    opt = torch.optim.Adam(model.parameters())
    for _ in range(n_warmup):
        opt.zero_grad()
        nn.functional.mse_loss(model(x_data), y_data).backward()
        opt.step()
    t0 = time.perf_counter()
    for _ in range(n_steps):
        opt.zero_grad()
        nn.functional.mse_loss(model(x_data), y_data).backward()
        opt.step()
    elapsed = time.perf_counter() - t0
    print(f"  {label:<25} {elapsed:.3f}s ({n_steps} steps)")
    return elapsed

t_eager = bench(make_model(), "Eager")

torch._dynamo.config.compiled_autograd = True
t_compiled = bench(torch.compile(make_model()), "Compiled + CA")
torch._dynamo.config.compiled_autograd = False

ratio = t_eager / t_compiled if t_compiled > 0 else float('inf')
print(f"\n  Speedup: {ratio:.2f}x")
print("  (On GPU, expect 1.2-1.5x speedup from kernel fusion and reduced launch overhead)")

## 15. Disabling and Debugging

Compiled autograd can be toggled at any point:

In [ ]:
# Enable
torch._dynamo.config.compiled_autograd = True
model = torch.compile(nn.Linear(8, 4))
loss = model(torch.randn(2, 8)).sum()
loss.backward()
print("With compiled autograd: OK")

# Disable
torch._dynamo.config.compiled_autograd = False
torch._dynamo.reset()
model2 = torch.compile(nn.Linear(8, 4))
loss = model2(torch.randn(2, 8)).sum()
loss.backward()
print("Without compiled autograd: OK")

## Exercise

**Use `aot_function` to inspect the forward/backward graphs of a 3-layer MLP and count saved tensors.**

Build a model: `relu(x @ W1 + b1) -> relu(h @ W2 + b2) -> h @ W3 + b3`

Questions:
1. How many tensors are saved from forward for backward?
2. Which ops appear in the backward graph?
3. Can you identify which tensors are saved vs recomputed?

In [ ]:
# Your solution here!

# Hint: use aot_function with custom fw_compiler and bw_compiler
# that print the graph and count outputs/inputs.

# def three_layer_mlp(x, w1, b1, w2, b2, w3, b3):
#     h = torch.relu(x @ w1 + b1)
#     h = torch.relu(h @ w2 + b2)
#     return (h @ w3 + b3).sum()
#
# ...
# compiled = aot_function(three_layer_mlp, fw_compiler=..., bw_compiler=...)
# ...

In [ ]:
# Solution

def three_layer_mlp(x, w1, b1, w2, b2, w3, b3):
    h = torch.relu(x @ w1 + b1)
    h = torch.relu(h @ w2 + b2)
    return (h @ w3 + b3).sum()

fw_saved = {}
bw_details = {}

def exercise_fw(gm, example_inputs):
    output_node = [n for n in gm.graph.nodes if n.op == 'output'][0]
    output_args = output_node.args[0]
    n_outputs = len(output_args) if isinstance(output_args, (list, tuple)) else 1
    fw_saved['count'] = n_outputs - 1  # subtract the model output
    
    print("FORWARD GRAPH:")
    for n in gm.graph.nodes:
        if n.op == 'call_function':
            name = n.target.__name__ if hasattr(n.target, '__name__') else str(n.target)
            print(f"  {n.name}: {name}")
    print(f"\nForward outputs: {n_outputs} (1 model output + {n_outputs - 1} saved)")
    return gm

def exercise_bw(gm, example_inputs):
    placeholders = [n for n in gm.graph.nodes if n.op == 'placeholder']
    call_nodes = [n for n in gm.graph.nodes if n.op == 'call_function']
    
    print(f"\nBACKWARD GRAPH:")
    print(f"  Inputs: {len(placeholders)} (grad_outputs + saved tensors)")
    print(f"  Ops:")
    for n in call_nodes:
        name = n.target.__name__ if hasattr(n.target, '__name__') else str(n.target)
        print(f"    {n.name}: {name}")
    return gm

compiled = aot_function(three_layer_mlp, fw_compiler=exercise_fw, bw_compiler=exercise_bw)

x = torch.randn(4, 8, requires_grad=True)
w1 = torch.randn(8, 6, requires_grad=True)
b1 = torch.randn(6, requires_grad=True)
w2 = torch.randn(6, 4, requires_grad=True)
b2 = torch.randn(4, requires_grad=True)
w3 = torch.randn(4, 3, requires_grad=True)
b3 = torch.randn(3, requires_grad=True)

out = compiled(x, w1, b1, w2, b2, w3, b3)
out.backward()

print(f"\nSaved tensors from forward: {fw_saved.get('count', '?')}")
print("Matmul results are saved; relu/add are recomputed.")

## Key Takeaways

1. **Eager backward is the bottleneck** — `torch.compile` alone only optimizes forward
2. **AOTAutograd traces both passes at compile time** — produces two FX graphs (forward + backward) that Inductor optimizes
3. **The joint graph** contains forward + backward ops; the partitioner splits them and decides what to save vs recompute
4. **min-cut partitioner** automates operator-level activation checkpointing — saves expensive ops, recomputes cheap ones
5. **Compiled Autograd** compiles the autograd engine itself — `torch._dynamo.config.compiled_autograd = True`
6. **Correctness is preserved** — compiled backward produces the same gradients as eager
7. **Debug with TORCH_LOGS** — `aot`, `output_code`, `compiled_autograd` for different levels of detail
8. **GPU gains are significant** (1.2-1.5x); CPU gains are modest

---

**Next**: This is the final module in the current series.

**Previous**: [Module 37 — torch.export Deep Dive](../37_export_deep_dive/)